# TOURASNA — Complete SW Team Integration Guide
## ML Systems Handoff Document

---

# PART 1 — AI Recommendation System

---

## What It Does

A deep learning recommendation engine that takes a user profile and interaction history and returns personalized Egyptian landmark recommendations. It supports two services: **DayPlan** (top 10 landmarks for one day) and **TripPlan** (5 landmarks per day for multi-day trips). Recommendations improve automatically with every like and dislike the user submits.

---

## Files SW Team Receives

```
model_inference.py                ← recommendation engine
utils.py                          ← feature engineering shared utilities
travel_recommendation_model.keras ← trained neural network
label_encoders.pkl                ← encoding artifacts
all_categories.pkl                ← category definitions
model_config.json                 ← model metadata and performance metrics
synthetic_user_landmarks.csv      ← landmark database
real_user_interactions.csv        ← feedback log (auto-created on first interaction)
```

> **All files must be in the same directory. Do not separate them.**

---

## Python Environment

```bash
pip install tensorflow pandas numpy scikit-learn
# Python 3.9+
```

---

## Database Tables SW Team Must Build

```sql
CREATE TABLE users (
    user_id           VARCHAR(50) PRIMARY KEY,
    user_age          INT,
    user_gender       VARCHAR(10),
    user_budget       VARCHAR(20),
    user_travel_type  VARCHAR(20),
    user_preferences  VARCHAR(500),
    interaction_count INT DEFAULT 0,
    created_at        DATETIME
);

CREATE TABLE user_interactions (
    id                INT AUTO_INCREMENT PRIMARY KEY,
    timestamp         DATETIME,
    user_id           VARCHAR(50),
    landmark_name     VARCHAR(200),
    landmark_category VARCHAR(100),
    event_type        VARCHAR(20),
    affinity_signal   FLOAT
);
```

---

## The Two Services

### DayPlan
Returns top 10 ranked landmarks for a single day.

### TripPlan
Returns exactly 5 landmarks per day organized by day number.

| trip_days | Total Landmarks |
|-----------|----------------|
| 1 | 5 |
| 2 | 10 |
| 3 | 15 |
| 7 | 35 |
| 14 | 70 (maximum) |

---

## API Endpoints SW Team Must Build

### `POST /api/recommendations`

#### DayPlan Request Body

```json
{
  "plan_type": "DayPlan",
  "user_id": "U1001",
  "interaction_count": 7,
  "user_age": 25,
  "user_gender": "Male",
  "user_budget": "medium",
  "user_travel_type": "solo",
  "user_preferences": [
    "Museum",
    "Pharaonic Site",
    "Islamic Monument",
    "Nile Cruise"
  ],
  "interaction_history": [
    { "category": "Museum",      "affinity_signal": 0.8  },
    { "category": "Nile Cruise", "affinity_signal": -0.3 }
  ]
}
```

#### TripPlan Request Body

```json
{
  "plan_type": "TripPlan",
  "trip_days": 3,
  "user_id": "U1001",
  "interaction_count": 7,
  "user_age": 25,
  "user_gender": "Male",
  "user_budget": "medium",
  "user_travel_type": "solo",
  "user_preferences": [
    "Museum",
    "Pharaonic Site",
    "Islamic Monument",
    "Nile Cruise"
  ],
  "interaction_history": [
    { "category": "Museum",      "affinity_signal": 0.8  },
    { "category": "Nile Cruise", "affinity_signal": -0.3 }
  ]
}
```

> **New user with no history — pass empty list:** `"interaction_history": []`

---

### `POST /api/feedback`

```json
{
  "user_id": "U1001",
  "landmark_name": "Mahmoud Khalil Museum",
  "event_type": "like"
}
```

```json
{
  "user_id": "U1001",
  "landmark_name": "Cafelluca Boat",
  "event_type": "dislike"
}
```

**Signal weights:**

| Event | Signal | Meaning |
|-------|--------|---------|
| `like` | +0.8 | Strong positive signal |
| `dislike` | -0.3 | Moderate negative signal |

---

## Field Validators

| Field | Valid Values |
|-------|-------------|
| `plan_type` | `"DayPlan"` or `"TripPlan"` only |
| `trip_days` | Integer 1–14, required for TripPlan only |
| `user_budget` | `"low"`, `"medium"`, `"high"` |
| `user_travel_type` | `"solo"`, `"couple"`, `"family"`, `"luxury"` |
| `user_preferences` | Minimum 1 from the 27 categories below |
| `user_age` | Integer 18–75 |
| `user_gender` | `"Male"` or `"Female"` |
| `interaction_count` | Integer starting at 0, never reset |
| `event_type` | `"like"` or `"dislike"` only |

### The 27 Valid Preference Categories

```
Activity, Ancient Monument, Antiques, Art Gallery,
Bazaar / Souq, Coptic Site, Cultural Center, Day Trip Site,
Escape Room, Food Tour, Gold & Jewelry Market, Horse Riding,
Islamic Monument, Landmark, Museum, Nature Reserve,
Nile Cruise, Nile View Restaurant, Park / Garden,
Pharaonic Site, Rooftop Restaurant, Shopping Mall,
Souvenir Shop, Sport & Recreation, Theme Park,
Traditional Restaurant, Zoo / Aquarium
```

---

## How To Call The Engine

```python
from model_inference import RecommendationEngine, validate_plan_input
from utils import validate_user_input
import pickle
from tensorflow import keras
import pandas as pd

# ================================================================
# LOAD ONCE AT SERVER STARTUP — NEVER PER REQUEST
# ================================================================
model = keras.models.load_model(
    'travel_recommendation_model.keras',
    compile=False
)
with open('all_categories.pkl', 'rb') as f:
    all_categories = pickle.load(f)

df = pd.read_csv('synthetic_user_landmarks.csv',
    usecols=['landmark_name', 'landmark_category',
             'landmark_budget', 'landmark_rate',
             'landmark_Suitable_Travel_Type'])
unique_landmarks = df.drop_duplicates()

engine = RecommendationEngine(
    model=model,
    all_categories=all_categories,
    unique_landmarks=unique_landmarks
)
# ================================================================

# ================================================================
# CALL ON EVERY RECOMMENDATION REQUEST
# ================================================================
def handle_recommendation_request(request_data):
    user_input = {
        'plan_type':        request_data['plan_type'],
        'user_age':         request_data['user_age'],
        'user_gender':      request_data['user_gender'],
        'user_budget':      request_data['user_budget'],
        'user_travel_type': request_data['user_travel_type'],
        'user_preferences': request_data['user_preferences'],
    }
    if request_data['plan_type'] == 'TripPlan':
        user_input['trip_days'] = request_data['trip_days']

    validate_user_input(user_input)
    validate_plan_input(user_input)

    top_n = 10 if request_data['plan_type'] == 'DayPlan' \
            else request_data['trip_days'] * 5

    recommendations = engine.get_recommendations(
        user_input=user_input,
        user_id=request_data['user_id'],
        interaction_count=request_data['interaction_count'],
        top_n=top_n,
        interaction_history=request_data.get('interaction_history', [])
    )
    return recommendations

# ================================================================
# CALL ON EVERY LIKE OR DISLIKE
# ================================================================
def handle_feedback_request(feedback_data):
    engine.log_interaction(
        user_id=feedback_data['user_id'],
        landmark_name=feedback_data['landmark_name'],
        event_type=feedback_data['event_type']
    )
    # SW team must ALSO:
    # 1. Save to user_interactions table in DB
    # 2. Save landmark_category with the record
    # 3. Increment interaction_count for this user in DB
```

---

## Response Formats

### DayPlan Response

```json
[
  {
    "name": "Mahmoud Khalil Museum",
    "category": "Museum",
    "rating": 4.6,
    "budget": "medium",
    "final_score": 0.909,
    "dl_score": 0.953,
    "pop_score": 0.790
  }
]
```

### TripPlan Response

```json
{
  "plan_type": "TripPlan",
  "trip_days": 3,
  "total_landmarks": 15,
  "days": [
    {
      "day": 1,
      "landmarks": [
        {
          "name": "Mahmoud Khalil Museum",
          "category": "Museum",
          "rating": 4.6,
          "score": 0.909
        }
      ]
    },
    { "day": 2, "landmarks": ["5 landmarks"] },
    { "day": 3, "landmarks": ["5 landmarks"] }
  ]
}
```

---

## How Personalization Works

| interaction_count | Blending |
|-------------------|---------|
| 0 | 80% Popularity / 20% DL (new user) |
| 10 | 67% Popularity / 33% DL |
| 20 | 50% Popularity / 50% DL |
| 30 | 40% Popularity / 60% DL |
| 55+ | 27% Popularity / 73% DL (experienced) |

**SW team must:**
- Store `interaction_count` per user in DB
- Increment by 1 after every like or dislike
- Always pass current count on every request
- Never reset to 0 unless account is deleted
- Fetch full `interaction_history` from DB and pass on every request

### How Personalization Grows

| Stage | Result |
|-------|--------|
| Visit 1 — 0 interactions | Profile-based, same as similar users |
| Visit 2 — 3+ interactions | Liked categories ranked higher, disliked lower |
| Visit 5 — 10+ interactions | Noticeably tailored to this individual |
| Visit 10 — 25+ interactions | Two users with identical surveys get completely different results |

---

## Budget Rules

| User Budget | Behaviour |
|-------------|-----------|
| `low` | Only low budget landmarks shown |
| `medium` | At least 8/10 medium budget, max 2 low budget, flagged `[LOWER BUDGET]`, no high budget shown |
| `high` | All budget levels eligible |

---

## TripPlan Day Ordering

> Landmarks within each day are **diversity-ordered**, not purely score-ordered. This is intentional — it prevents 5 museums in one day. **Do NOT report this as a bug.**

---

## Performance Notes

> ⚠️ **Load model ONCE at server startup.** Never load per request — takes ~3 seconds.

> ⚠️ **CPU only on Windows.** Deploy on Linux for production performance. Inference ~1–2 seconds per request on CPU.

> ⚠️ **All 7 files must stay in the same directory.**

> ⚠️ **`interaction_count` must never reset to 0** unless user account is deleted.

> ⚠️ **Always pass `interaction_history` on every request.** Use `[]` for new users with no history.

---

## QA Test Cases — Recommendation System

### Test 1 — New User DayPlan
```
Input:    interaction_count=0, interaction_history=[]
Expected: 10 landmarks
Expected: Globally popular landmarks dominate
```

### Test 2 — Experienced User DayPlan
```
Input:    interaction_count=55
          interaction_history=[
            {"category":"Museum","affinity_signal":0.8},
            {"category":"Museum","affinity_signal":0.8}
          ]
Expected: Museums ranked higher than Test 1
Expected: Different ranking than new user
```

### Test 3 — TripPlan 3 Days
```
Input:    plan_type="TripPlan", trip_days=3
Expected: 15 landmarks total
Expected: Exactly 5 per day
Expected: DAY 1, DAY 2, DAY 3 structure
```

### Test 4 — Personalization Proof ✅
```
User A history: liked 3 Museums, disliked Nile Cruise
User B history: liked 3 Nile Cruises, disliked Museums
Same age, gender, budget, travel type, preferences

Expected: User A top 3 = Museums
Expected: User B top 3 = Nile Cruises
Expected: Completely different lists
→ If this passes, real personalization is confirmed
```

### Test 5 — Invalid plan_type
```
Input:    plan_type="WeekPlan"
Expected: Clear error message
```

### Test 6 — Budget Constraint
```
Input:    user_budget="medium"
Expected: Maximum 2 low budget items
Expected: [LOWER BUDGET] flag visible
```

### Test 7 — Invalid event_type
```
Input:    event_type="click"
Expected: Clear error — "Accepted: like, dislike"
```

---

# PART 2 — FAHMY Chatbot

---

## What It Does

A locally-deployed RAG-powered Egyptian tourism chatbot named Fahmy. It uses a FAISS vector database for semantic search, a locally-running Mistral-7B language model via Ollama, and supports 16 languages with streaming responses. It finds landmarks based on meaning, not just keywords.

---

## Files SW Team Receives

```
fahmy_chatbot.py            ← main chatbot engine
landmarksANDplaces_info.csv ← landmark knowledge base
knowledge_base.faiss        ← pre-built FAISS index
embeddings.pkl              ← pre-computed embeddings
```

> The FAISS index and embeddings are generated automatically on first run. They only need regeneration if the landmark dataset changes.

---

## Prerequisites

```bash
pip install faiss-cpu numpy pandas requests
# Python 3.9+

# Ollama — local LLM runtime
# Download from: https://ollama.com

# Pull both required models
ollama pull mistral:7b
ollama pull nomic-embed-text

# Verify both are available
ollama list
```

> **If `nomic-embed-text` is not pulled, the chatbot falls back to keyword search and RAG will not function. A clear warning is printed at startup.**

---

## How To Run

```bash
# Standard run
python fahmy_chatbot.py

# Force rebuild FAISS index (use when dataset changes)
python fahmy_chatbot.py --rebuild-index

# Custom Ollama URL (if not on localhost)
python fahmy_chatbot.py http://your-server:11434

# Demo mode
python fahmy_chatbot.py --demo
```

---

## API Endpoint SW Team Must Build

### `POST /api/chat`

**Request:**
```json
{
  "user_id": "U1001",
  "message": "Tell me about museums in Cairo",
  "session_id": "session_abc123"
}
```

**Response:**
```json
{
  "response": "Oh, Cairo has some incredible museums!...",
  "language_detected": "english",
  "landmarks_retrieved": [
    {
      "name": "Egyptian Museum",
      "category": "Museum",
      "similarity_score": 0.87
    }
  ],
  "session_id": "session_abc123"
}
```

---

## How To Call The Engine

```python
from fahmy_chatbot import EgyptianTourismChatbot

# ================================================================
# ONE INSTANCE PER USER SESSION — Initialize at session start
# ================================================================
sessions = {}

def get_or_create_session(session_id):
    if session_id not in sessions:
        sessions[session_id] = EgyptianTourismChatbot()
    return sessions[session_id]

def end_session(session_id):
    if session_id in sessions:
        del sessions[session_id]

# ================================================================
# NON-STREAMING — Simple request/response
# ================================================================
def handle_chat_request(session_id, user_message):
    chatbot = get_or_create_session(session_id)
    return chatbot.process_query(user_message)

# ================================================================
# STREAMING — Recommended for mobile UI
# ================================================================
def handle_chat_stream(session_id, user_message):
    chatbot = get_or_create_session(session_id)
    for chunk in chatbot.process_query_stream(user_message):
        if chunk:
            send_to_client(chunk)  # SW implements this
```

---

## 16 Supported Languages

```
Arabic, Hebrew, Persian/Farsi, Greek, Russian,
Chinese, Japanese, Korean, Thai, Hindi,
French, Spanish, German, Italian, Portuguese, English
```

> **SW team does not pass a language parameter.** Detection is fully automatic inside the engine.

| User Language | Chatbot Response Language |
|---------------|--------------------------|
| Arabic | Arabic |
| Chinese | Chinese |
| Russian | Russian |
| French | French |
| Any of 16 supported | Same language |

---

## How Language Detection Works

**Priority order (highest confidence first):**

| Tier | Languages | Detection Method |
|------|-----------|-----------------|
| Script-based | Arabic, Hebrew, Persian, Greek, Russian, Korean, Japanese, Chinese, Thai, Hindi | Single character sufficient |
| Lexical-based | French, Spanish, German, Italian, Portuguese | Keyword matching with adaptive threshold |
| Default | English | Fallback |

> **Do not attempt to override language detection from the frontend.** Detection runs inside the engine before the LLM generates anything.

---

## Streaming Behavior

```python
# Backend — stream chunks via WebSocket or SSE
for chunk in chatbot.process_query_stream(user_input):
    if chunk:
        websocket.send(chunk)

# Frontend — append chunks as they arrive
# Do not wait for full response before displaying
```

> **Recommended:** Server-Sent Events (SSE) or WebSocket for real-time delivery.

---

## How RAG Works On Each Request

```
User sends message
        ↓
Language detected
        ↓
Query encoded into 768-dimensional semantic vector
        ↓
FAISS index searched — top 5 similar landmarks
(cosine similarity, threshold 0.3)
        ↓
Diversity filter — max 2 per category
        ↓
If FAISS returns nothing → keyword search fallback
        ↓
Augmented prompt built
(facts + last 3 conversation turns + current query)
Token budget: 2500 tokens maximum
        ↓
Language instruction injected into system prompt
        ↓
Mistral-7B generates complete response
        ↓
Language consistency validated
(max 1 silent retry if wrong language)
        ↓
Validated response streamed character by character
```

---

## Session Management

```python
sessions = {}

def get_or_create_session(session_id):
    if session_id not in sessions:
        sessions[session_id] = EgyptianTourismChatbot()
    return sessions[session_id]

def end_session(session_id):
    if session_id in sessions:
        del sessions[session_id]
```

> Session history is limited to the last **8 conversation turns** automatically.

---

## Fallback Behavior

| Scenario | Behaviour |
|----------|-----------|
| `nomic-embed-text` not pulled | Keyword search instead of FAISS, quality reduced, warning printed, all 16 languages still work |
| Ollama completely offline | Pre-written fallback responses in user's language, landmark data from keyword search, no LLM generation, still multilingual |

---

## Performance Notes

> ⚠️ **Start Ollama BEFORE running the chatbot:** `ollama serve`

> ⚠️ **First run builds FAISS index.** Takes 2–5 minutes for 677 landmarks. Subsequent runs load from disk in under 1 second.

> ⚠️ **Response latency: 2–5 seconds on CPU.** Streaming masks this — user sees typing immediately.

> ⚠️ **Memory: ~6 GB RAM during inference.** Plan server resources accordingly.

> ⚠️ **One chatbot instance per user session.** Not one global shared instance.

---

## QA Test Cases — Chatbot

### Test 1 — English response
```
Input:    "hello"
Expected: Response in English only
Expected: Streaming character by character
```

### Test 2 — Arabic response
```
Input:    "مرحباً"
Expected: Response entirely in Arabic, no English words
```

### Test 3 — French response
```
Input:    "Bonjour"
Expected: Response entirely in French
```

### Test 4 — Spanish response
```
Input:    "Hola, ¿cómo estás?"
Expected: Response entirely in Spanish
```

### Test 5 — Russian response
```
Input:    "Привет"
Expected: Response entirely in Russian
```

### Test 6 — Chinese response
```
Input:    "你好"
Expected: Response entirely in Chinese
```

### Test 7 — Semantic RAG proof
```
Input:    "I want to see something ancient and historical"
Expected: Response mentions specific landmark names from the dataset
Expected: Not a generic response
Expected: Proves semantic search works beyond keyword matching
```

### Test 8 — Language-aware farewell
```
English session: type "exit"
Expected: 🏛️ Safe travels, my friend!

Arabic session: type "exit"
Expected: مع السلامة يا صديقي!

French session: type "exit"
Expected: Au revoir mon ami!
```

### Test 9 — Fallback when Ollama offline
```
Action:   Stop Ollama, send any query
Expected: Fallback response in user's language
Expected: Does not crash
```

### Test 10 — Session isolation
```
Action:   Session A discusses museums
          Session B discusses beaches
Expected: Each maintains independent history
Expected: No cross-contamination between sessions
```

---

# PART 3 — How Both Systems Connect In The App

---

## The User Flow

```
User opens app
        ↓
Registers / logs in (Firebase — SW team handles)
        ↓
Fills profile survey:
  age, gender, budget, travel type, preferences
        ↓
SW team stores profile in users table
        ↓
        ┌─────────────────────────────────┐
        │                                 │
        ▼                                 ▼
RECOMMENDATION ENGINE               FAHMY CHATBOT
DayPlan / TripPlan                  Tourism Q&A
        │                                 │
        ▼                                 ▼
User sees landmark list          User chats naturally
        │                                 │
        ▼                                 ▼
User likes / dislikes            Chatbot retrieves
landmark from results            relevant landmarks
        │                         using RAG + FAISS
        ▼                                 │
SW saves to DB                           ▼
Increments interaction_count    Responds in user's
        │                         language naturally
        ▼
Next recommendation request
includes updated history
→ Better personalization
```

---

## How The Two Systems Share Data

The two systems do **not** directly communicate with each other. SW team is the bridge between them.

| System | Role |
|--------|------|
| Recommendation Engine | Produces landmark lists |
| Fahmy Chatbot | Answers questions about landmarks |

**If user asks Fahmy about a recommended landmark:**
- SW team passes the landmark name in the chat message
- Fahmy retrieves it from FAISS knowledge base
- Fahmy answers in the user's language

**If user likes a landmark Fahmy mentioned:**
- SW team calls `POST /api/feedback`
- Recommendation engine updates affinity
- Next DayPlan / TripPlan reflects this preference

---

## Complete SW Team Responsibility Summary

### Authentication
- Firebase authentication (SW team builds)
- JWT tokens on all API requests

### User Profile
- Store in `users` table
- Pass to recommendation engine on every request

### Interaction History
- Save every like/dislike to `user_interactions` table
- Include `landmark_category` in every record
- Increment `interaction_count` after every event
- Fetch full history and pass on every recommendation request

### Session Management (Chatbot)
- One `EgyptianTourismChatbot` instance per user session
- Create on session start, destroy on session end
- Never share instance across users

### Frontend Streaming (Chatbot)
- Implement WebSocket or SSE
- Append chunks as they arrive
- Do not wait for complete response

### Language Handling
- Do not pass language to chatbot
- Detection is fully automatic
- Do not override from frontend

### Error Handling
- Handle Ollama being offline gracefully
- Handle recommendation engine errors gracefully
- Never expose internal error messages to users

---

## One-Paragraph Summary For SW Team

> "The TOURASNA ML systems consist of two independent engines. The **Recommendation Engine** takes a user profile and interaction history and returns a ranked list of Egyptian landmarks for DayPlan or TripPlan — call it on every recommendation request and pass the full interaction history from your database so personalization improves with every like and dislike. The **Fahmy Chatbot** is a RAG-powered conversational assistant that automatically detects the user's language, retrieves semantically relevant landmarks using FAISS, and responds through Ollama's Mistral-7B model — create one instance per user session, stream chunks via WebSocket or SSE, and let the engine handle all language detection internally. SW team is responsible for authentication, database management, session isolation, interaction history storage, and connecting the two systems through the user interface."

---

## ML Team Contact Points

| Trigger | Action |
|---------|--------|
| Integration questions about either engine | ML team answers |
| Dataset changes requiring FAISS rebuild | ML team advises on `--rebuild-index` usage |
| 500–1000 real user interactions collected | ML team retrains recommendation model, delivers new `.keras` file, SW team redeploys |
| New landmarks added to dataset | ML team updates both CSV files and provides new `knowledge_base.faiss` and `embeddings.pkl`, SW team replaces files and restarts |

============================================================================================================================================================

# TOURASNA — Old vs Final System Comparison
## Recommendation System & Chatbot Evolution Report

---

# SECTION 1 — AI Recommendation System

---

## Summary

| | V1 — Original | V2 — Final |
|---|---|---|
| Label type | Discrete rule-based | Continuous weighted formula |
| Personalization | Zero — rule matching only | Real per-user behavioral learning |
| Primary metrics | MSE / R² (regression) | NDCG@5 / Precision@5 / MRR (ranking) |
| Services | Single output only | DayPlan + TripPlan |
| Feedback system | None | Like / Dislike with affinity signals |
| Cold start | Not handled | Popularity blending |
| Score range | 5–6 discrete bands | Continuous 0.085 → 1.000 |
| Learning from behavior | Never | Every interaction |

---

## 1.1 Label Generation

### V1 — The Problem
Labels were generated using a simple rule-based formula:

```python
# V1 label generation (simplified)
if budget_match AND category_match AND travel_type_match:
    score = 1.0
elif partial_match:
    score = 0.68  # or 0.60, 0.40, 0.33
else:
    score = 0.0
```

This produced only 5–6 discrete score values: `0.0, 0.33, 0.40, 0.60, 0.68, 1.0`. The neural network learned to reconstruct these rules, not genuine user preferences. It was effectively a rule engine dressed as AI.

**Evidence from V1 predicted vs actual plot:**
- Horizontal discrete bands instead of a diagonal cloud
- Every matching user-landmark pair scored exactly `1.000`
- The model could not differentiate between two landmarks that both matched the rules

### V2 — The Fix
Labels replaced with a weighted continuous formula:

```python
score = (category_affinity  × 0.35) +
        (rating_normalized  × 0.25) +
        (budget_fit         × 0.20) +
        (travel_type_fit    × 0.10) +
        (popularity_trend   × 0.10) +
        gaussian_noise(0, 0.05)
```

**Result:**
- Label mean: 0.795, std: 0.150, range: 0.085 → 1.000
- Genuine continuous distribution
- Model forced to learn real tradeoffs between features

---

## 1.2 Evaluation Metrics

### V1 — The Problem
MSE and R² were used as primary metrics. These measure regression accuracy, not recommendation quality. A model that predicts exact rule values scores perfectly on these metrics while being useless as a recommender.

| V1 Metric | V1 Score | What It Actually Meant |
|---|---|---|
| R² | 0.9422 | Model perfectly reconstructed the rules |
| Test MSE | 0.004119 | Predicted the formula output precisely |
| Test Accuracy | 98.95% | Binary threshold of 0.5 was trivially easy |

### V2 — The Fix
Replaced with proper recommendation ranking metrics:

| V2 Metric | V2 Score | What It Actually Means |
|---|---|---|
| NDCG@5 | 0.9536 | Top 5 items are in near-perfect relevance order |
| Precision@5 | 0.6243 | More than 3 of every 5 recommendations are genuinely relevant |
| MRR | 0.4295 | First truly relevant item appears at rank 2–3 on average |
| R² | 0.8020 | Honest score on a harder continuous task |

---

## 1.3 Inference Output Quality

### V1 — The Problem
```
Mahmoud Khalil Museum       → Score: 1.000  ← fake
Panorama October 6 War      → Score: 1.000  ← fake
Children's Museum           → Score: 1.000  ← fake
Pharaonic Museum of Papyrus → Score: 1.000  ← fake
Andalusia Park              → Score: 0.679  ← fake
```
Nine of ten recommendations scored exactly `1.000`. No meaningful differentiation between landmarks.

### V2 — The Fix
```
Mahmoud Khalil Museum       → Score: 0.909  ← real
Umm Kulthum Museum          → Score: 0.885  ← real
Panorama October 6 War      → Score: 0.870  ← real
Pharaonic Museum of Papyrus → Score: 0.945  ← real
Andalusia Park              → Score: 0.845  ← real
```
Every landmark has a unique, meaningful score reflecting genuine compatibility with the user profile.

---

## 1.4 Personalization

### V1 — The Problem
Two completely different users with the same survey answers received **100% identical recommendations** because the model only matched rules, not individuals.

```
User A: 30yo, Male, Medium budget, Solo, [Museum, Pharaonic]
User B: 30yo, Male, Medium budget, Solo, [Museum, Pharaonic]
Result: Identical lists — forever
```

### V2 — The Fix
Individual affinity vectors with behavioral learning:

**Affinity vector generation:**
Each user receives a unique 27-dimensional continuous affinity vector, even with identical survey answers, through controlled Gaussian variance.

**Interaction history read-back:**
```python
# Each like/dislike updates the affinity vector
like    → +0.8 affinity (strong positive signal)
dislike → -0.3 affinity (moderate negative signal)
```

**Personalization proof from testing:**
```
User A: liked 3 Museums, disliked Nile Cruise
→ Top 3: Umm Kulthum Museum, Mahmoud Khalil Museum,
         Panorama of the October 6 War

User B: liked 3 Nile Cruises, disliked Museums
→ Top 3: Cafelluca Boat, Felucca Ride, Emo Tours Egypt

Same profile. Completely different results. ✅
```

---

## 1.5 Cold Start Handling

### V1 — The Problem
No cold start strategy existed. New users received the same output as any other user. There was no distinction between someone using the app for the first time and someone with 100 interactions.

### V2 — The Fix
Gradual blending strategy based on interaction count:

| interaction_count | Blending |
|---|---|
| 0 | 80% Popularity / 20% DL |
| 10 | 67% Popularity / 33% DL |
| 20 | 50% Popularity / 50% DL |
| 30 | 40% Popularity / 60% DL |
| 55+ | 27% Popularity / 73% DL |

New users are guided by global popularity until the system has enough behavioral signal to personalize confidently. This is the same approach used by Netflix and Spotify.

---

## 1.6 Services Architecture

### V1 — The Problem
Only a single flat output of 10 recommendations existed. No concept of multi-day planning, no service differentiation.

### V2 — The Fix

**DayPlan:** Top 10 landmarks for a single day.

**TripPlan:** Exactly 5 landmarks per day for any number of days (1–14):
```
trip_days = 1  →  5 landmarks
trip_days = 3  → 15 landmarks
trip_days = 7  → 35 landmarks
trip_days = 14 → 70 landmarks
```

---

## 1.7 Feedback System

### V1 — The Problem
No feedback mechanism. User behavior was never captured. The model had no way to learn from how users actually interacted with recommendations.

### V2 — The Fix
Complete like/dislike feedback loop:

```
User likes a landmark    → +0.8 affinity signal logged
User dislikes a landmark → -0.3 affinity signal logged
```

Every interaction is written to `real_user_interactions.csv` with:
- Timestamp
- User ID
- Landmark name and category
- Event type and affinity signal

On the next recommendation request, this history is read back and applied to the user's affinity vector, making every subsequent recommendation more accurate than the last.

---

## 1.8 Learning Curve Analysis

### V1 — The Problem
The training curve showed a suspicious flat pattern for 20 epochs followed by a sudden drop at epoch 20. This was not genuine learning — it was a learning rate artifact. The model was not getting smarter; the optimizer was just taking smaller steps.

### V2 — The Fix
Smooth, consistent improvement across all 30 epochs:

| Checkpoint | Val MAE | Val MSE | LR |
|---|---|---|---|
| Epoch 8 | 0.0520 | 0.0049 | 2.6e-04 |
| Epoch 16 | 0.0511 | 0.0048 | 1.5e-04 |
| Epoch 21 | 0.0507 | 0.0047 | 7.5e-05 |
| Epoch 26 | 0.0501 | 0.0045 | 2.0e-05 |

No cliff. No artifact. Every epoch improved over the previous one.

---

## 1.9 Complete Metrics Comparison

| Metric | V1 | V2 | Change |
|---|---|---|---|
| NDCG@5 | 0.8240 | **0.9536** | +15.7% |
| Precision@5 | 0.2811 | **0.6243** | +122% |
| MRR | 0.2399 | **0.4295** | +79% |
| R² | 0.9422 | **0.8020** | More honest |
| Overfitting ratio | 1.18x | **1.17x** | Maintained |
| Label type | 6 discrete bands | **Continuous** | Fundamental fix |
| Score range | 0.0 / 0.68 / 1.0 | **0.085 → 1.000** | Real distribution |
| Learning curve | Cliff at epoch 20 | **Smooth all 30 epochs** | No artifacts |
| Personalization | Zero | **Individual affinity vectors** | Real feature |
| Cold start | Not handled | **Popularity blending** | Production ready |
| Feedback loop | None | **Like/dislike logging** | Production ready |
| Services | Single output | **DayPlan + TripPlan** | Complete |

---

# SECTION 2 — FAHMY Chatbot

---

## Summary

| | V1 — Original | V2 — Final |
|---|---|---|
| Search method | Keyword matching | FAISS semantic search |
| Embedding model | None | nomic-embed-text-v1.5 (768 dimensions) |
| Languages supported | 3 (Arabic, English, French) | 16 |
| Language detection | Fragile keyword list | Unicode + lexical two-tier system |
| Language injection | After generation (too late) | Before generation (in system prompt) |
| Streaming validation | Stream first, validate after | Validate first, stream after |
| Wrong language fix | Silent — user never sees it | 1 silent retry, user sees correct version |
| Curly brace safety | None — crash risk | Escaped before .format() call |
| Index persistence | None — rebuilt every run | Saved to disk, loaded in under 1 second |
| Fallback behavior | English only | All 16 languages |
| Farewell messages | Random language | Matches user's conversation language |

---

## 2.1 Knowledge Retrieval — The Core Upgrade

### V1 — The Problem
Keyword matching only. The system searched for exact word matches between the user query and landmark text.

```python
# V1 search — keyword only
if query in landmark['name'].lower(): score += 3
if query in landmark['city'].lower(): score += 2
if query in landmark['subcategory'].lower(): score += 1
```

**Consequence:** Searching "ancient sites" would not find "Pharaonic temples." Searching "place to relax by water" would not find "Nile Cruise." The user had to know the exact words used in the database.

### V2 — The Fix
True FAISS semantic search with nomic-embed-text-v1.5 embeddings:

```
Every landmark → 768-dimensional semantic vector
Every query    → 768-dimensional semantic vector
Match          → Cosine similarity (FAISS IndexFlatIP)
Threshold      → 0.3 minimum similarity score
```

**Consequence:** Searching "ancient things" finds Pharaonic sites. Searching "something relaxing near water" finds Nile Cruises. The system understands meaning, not just words.

**Proven in testing:**
- Query: `"I want to see something ancient and historical"` → Retrieved specific Pharaonic and ancient sites not mentioned by name
- Query: `"أين أجد المعابد الفرعونية؟"` → Retrieved relevant Pharaonic temples in Arabic

---

## 2.2 Language Detection

### V1 — The Problem
A small hardcoded keyword list with a single Arabic character range check:

```python
# V1 detection — fragile
arabic_chars = sum(1 for c in text if '\u0600' <= c <= '\u06FF')
if arabic_chars > 0: return 'arabic'
french_markers = ['je', 'tu', 'nous', 'vous'...]
if french_count >= 2: return 'french'
return 'english'  # default
```

**Critical failure:** Typing "hello" caused the chatbot to respond in Arabic. The detector was only called *after* generation, so the LLM received no language instruction before generating a single token.

### V2 — The Fix
Two-tier detection system covering 16 languages:

**Tier 1 — Script-based (single character sufficient):**

| Language | Unicode Block |
|---|---|
| Arabic | U+0600–U+06FF |
| Hebrew | U+0590–U+05FF |
| Persian | U+FB50–U+FDFF |
| Greek | U+0370–U+03FF |
| Russian | U+0400–U+04FF |
| Korean | U+AC00–U+D7A3 |
| Japanese | U+3040–U+30FF |
| Chinese | U+4E00–U+9FFF |
| Thai | U+0E00–U+0E7F |
| Hindi | U+0900–U+097F |

**Tier 2 — Lexical (adaptive threshold):**

| Input Length | Keyword Matches Required |
|---|---|
| ≤3 words | 1 match sufficient |
| 4–12 words | 2 matches required |
| 13+ words | 3 matches required |

**Critical fix:** Language is now detected *before* generation and injected into the system prompt so the LLM knows which language to use before generating anything.

---

## 2.3 Streaming Architecture

### V1 — The Problem
Stream first, validate after:

```
Generate response → Stream to user → Validate language
                                            ↓
                              If wrong: generate corrected version
                                            ↓
                              Store corrected in memory only
                              User already saw wrong version
```

The user always saw the wrong-language response. The correction was invisible.

### V2 — The Fix
Validate first, stream after:

```
Generate response (non-streaming)
        ↓
Validate language consistency
        ↓
If wrong: retry once with explicit language instruction
          (maximum 1 retry, never loops, completely silent)
        ↓
Stream the validated response character by character
        ↓
User sees correct language from the first character
```

---

## 2.4 Language Support Expansion

### V1 — 3 Languages
Arabic, English, French only.

### V2 — 16 Languages

| Script Group | Languages |
|---|---|
| Arabic script | Arabic, Hebrew, Persian/Farsi |
| Other non-Latin | Greek, Russian, Chinese, Japanese, Korean, Thai, Hindi |
| Latin script | French, Spanish, German, Italian, Portuguese |
| Default | English |

All 16 languages are supported across:
- Response generation
- Farewell messages (language-matched)
- Fallback responses when Ollama is offline
- Interest extraction (city names in all scripts)
- Farewell loop exit detection in `main()`

---

## 2.5 Farewell System

### V1 — The Problem
Random selection from a small pool of English and Arabic farewell messages. An English user could receive a Portuguese farewell.

```python
# V1 farewell — random
farewells = [
    "🏛️ Safe travels...",       # English
    "مع السلامة يا صديقي!",      # Arabic
]
farewell = random.choice(farewells)  # Random — no language matching
```

### V2 — The Fix
Language-aware farewell that detects the last user message:

```python
# V2 farewell — language-aware
last_user_lang = 'english'
for msg in reversed(self.conversation.history):
    if msg.get('role') == 'user':
        last_user_lang = self.detect_language(msg.get('content', ''))
        break

farewell = farewell_map.get(last_user_lang, farewell_map['english'])
```

**Result:**
- English session → `🏛️ Safe travels, my friend!`
- Arabic session → `مع السلامة يا صديقي! مصر دايماً مستنياك`
- French session → `Au revoir mon ami! L'Égypte vous attend toujours!`
- And so on for all 16 languages

---

## 2.6 Safety and Stability

### V1 — The Problem
Multiple crash risks:

1. **Curly brace crash:** If any landmark description in the CSV contained `{` or `}`, Python's `.format()` call would raise a `KeyError` and crash the chatbot silently
2. **No embedding model check:** If `nomic-embed-text` was not pulled, errors were silent and confusing
3. **No atomic index saves:** If the build process was interrupted, a corrupted partial index could be saved

### V2 — The Fix

1. **Curly brace safety:**
```python
context_safe = context.replace('{', '{{').replace('}', '}}')
```

2. **Embedding model check:**
```python
if not embed_found:
    print(f"Warning: Embedding model not found. Run: ollama pull {self.embedding_model}")
    self.embedding_available = False
```

3. **Atomic index saves:**
```python
# Write to temp files first
faiss.write_index(self.faiss_index, FAISS_INDEX_PATH + '.tmp')
# Only rename to final after both files are complete
os.replace(tmp_faiss, FAISS_INDEX_PATH)
os.replace(tmp_pkl, EMBEDDINGS_PKL_PATH)
```

---

## 2.7 Fallback Language Support

### V1 — The Problem
When Ollama was offline, all fallback responses were hardcoded in English regardless of the user's language:

```python
# V1 fallback — English only
return "Hey there! 😊 I'm Fahmy, your Egyptian tourism friend."
return "🏛️ Egypt has so many amazing places!"
```

An Arabic-speaking user with no internet connection would receive English responses.

### V2 — The Fix
All fallback responses are language-aware using the same `detect_language()` method:

```python
# V2 fallback — language-aware
user_lang = self.detect_language(user_input)
fallback_map = {
    'arabic':  'مرحباً! 😊 أنا فهمي، صديقك في السياحة المصرية.',
    'french':  'Bonjour! 😊 Je suis Fahmy, votre ami touristique.',
    'spanish': '¡Hola! 😊 Soy Fahmy, tu amigo de turismo egipcio.',
    # ... all 16 languages
    'english': "Hey there! 😊 I'm Fahmy, your Egyptian tourism friend."
}
return fallback_map.get(user_lang, fallback_map['english'])
```

---

## 2.8 Performance Comparison

| Metric | V1 | V2 |
|---|---|---|
| Retrieval method | Keyword scoring | FAISS cosine similarity |
| Retrieval accuracy (Precision@5) | Not measured | **0.83** |
| MRR | Not measured | **0.79** |
| NDCG | Not measured | **0.81** |
| Languages | 3 | **16** |
| Language detection accuracy | Fails on short inputs | **Robust — single char sufficient** |
| Wrong language responses | Frequent | **Max 1 silent retry** |
| Index build time | Every startup | **Once — then <1 second** |
| Crash risk (curly braces) | Yes | **Eliminated** |
| Farewell language matching | Random | **Always correct** |
| Fallback language matching | English only | **All 16 languages** |

---

# SECTION 3 — Architecture Evolution Diagram

```
┌─────────────────────────────────────────────────────────────────────┐
│                          V1 SYSTEMS                                 │
├──────────────────────────────┬──────────────────────────────────────┤
│   RECOMMENDATION SYSTEM      │   CHATBOT                           │
│                              │                                      │
│  User Profile                │  User Query                         │
│       ↓                      │       ↓                             │
│  Rule Matching               │  Keyword Search                     │
│  (if/else logic)             │  (exact word match)                 │
│       ↓                      │       ↓                             │
│  Discrete Score              │  Context Stuffed                    │
│  (1.0 or 0.68 etc.)          │  Into Prompt                        │
│       ↓                      │       ↓                             │
│  Static Ranked List          │  Stream Response                    │
│  (identical for all          │  (validate after —                  │
│   matching profiles)         │   user sees wrong lang)             │
│                              │                                      │
│  No feedback loop            │  3 languages                        │
│  No cold start               │  No FAISS index                     │
│  No learning                 │  No persistence                     │
└──────────────────────────────┴──────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────┐
│                          V2 SYSTEMS                                 │
├──────────────────────────────┬──────────────────────────────────────┤
│   RECOMMENDATION SYSTEM      │   CHATBOT                           │
│                              │                                      │
│  User Profile +              │  User Query                         │
│  Interaction History         │       ↓                             │
│       ↓                      │  Language Detected                  │
│  Affinity Vector             │  (before generation)                │
│  (continuous, unique         │       ↓                             │
│   per individual)            │  FAISS Semantic Search              │
│       ↓                      │  (meaning-based)                    │
│  Neural Network              │       ↓                             │
│  + Popularity Blend          │  Augmented Prompt                   │
│  (cold start aware)          │  (token budget managed)             │
│       ↓                      │       ↓                             │
│  Continuous Scores           │  Generate + Validate                │
│  (0.085 → 1.000)             │  (silent retry if wrong)            │
│       ↓                      │       ↓                             │
│  DayPlan / TripPlan          │  Stream Correct Response            │
│  (5 per day)                 │  (character by character)           │
│       ↓                      │                                      │
│  Like / Dislike              │  16 languages                       │
│  Feedback Logged             │  FAISS index persisted              │
│       ↓                      │  Language-aware farewell            │
│  Next visit more             │  Language-aware fallback            │
│  personalized                │                                      │
└──────────────────────────────┴──────────────────────────────────────┘
```

---

# SECTION 4 — One-Sentence Summary Per Change

| Change | Before | After |
|---|---|---|
| Label generation | Discrete rules → fake scores | Weighted continuous formula |
| Predicted vs actual | Horizontal bands | Diagonal cloud |
| Primary metric | MSE / R² | NDCG@5 / Precision@5 / MRR |
| Personalization | Two same-profile users = identical list | Two same-profile users = genuinely different lists |
| Cold start | No handling | Popularity → DL blending |
| Feedback | Nothing logged | Like/dislike with affinity signals |
| Learning over time | Never | Every interaction improves next result |
| Services | One flat list | DayPlan + TripPlan (5 per day) |
| Search method | Exact keyword match | FAISS cosine similarity on 768-dim vectors |
| Language detection | Fails on "hello" | Single character sufficient for non-Latin |
| Language injection | After generation (too late) | Before generation (in system prompt) |
| Streaming validation | Stream wrong, fix silently | Validate first, stream correct version |
| Languages | 3 | 16 |
| Farewell | Random language | Matches user's conversation language |
| Fallback | English only | All 16 languages |
| Index persistence | Rebuilt every startup | Saved once, loaded in under 1 second |
| Crash on curly braces | Yes | Eliminated by escaping |
| Atomic saves | No | Temp files → os.replace |

============================================================================================================================================================

# 🏛️ Handover Guide: AI Tourism Stack v2.0 (Ai-grad-2)

This project consists of two core systems: a **Multilingual RAG Chatbot (Fahmy)** and a **Personalized Travel Recommendation System**. Both are modular and ready for integration into a mobile or web application.

---

## 1. Infrastructure Requirements (Critical)

The **Chatbot** requires a local instance of [Ollama](https://ollama.ai/) to be running on the host machine/server.

**Steps for SW Team:**

1. Install Ollama on the server.
2. Pull the necessary models:

   ```bash
   ollama pull mistral:7b
   ollama pull nomic-embed-text
   ```

3. Ensure the API is accessible at `http://localhost:11434` (configurable in `fahmy_chatbot.py` -> `CONFIG`).

---

## 2. File Manifest

### A. Chatbot (RAG System)

*Located in `/Ollama_chatbot/`*
*(Note: Rename the folder from `Ollama-chatbot` to `Ollama_chatbot` to allow Python imports)*

- `fahmy_chatbot.py`: The main logic class `EgyptianTourismChatbot`.
- `knowledge_base.faiss`: Pre-built semantic index of 677+ landmarks.
- `embeddings.pkl`: Vectorized representations for fast retrieval.
- `landmarksANDplaces_info.csv`: The source dataset for RAG.

### B. Recommender (DL System)

*Located in `/RecommendationSystem/`*

- `model_inference.py`: The inference engine class `RecommendationEngine`.
- `travel_recommendation_model.keras`: The trained Deep Learning model.
- `label_encoders.pkl` & `all_categories.pkl`: Encoders for feature processing.
- `model_config.json`: Model hyperparameters and metadata.
- `utils.py`: Shared utilities for feature engineering.

---

## 3. Integration Guide

### How to use the Chatbot

Initialize the class once and call the streaming query method.

```python
from Ollama_chatbot.fahmy_chatbot import EgyptianTourismChatbot

chatbot = EgyptianTourismChatbot()

# For streaming responses (recommended for UI):
for chunk in chatbot.process_query_stream("Hello! Tell me about Cairo."):
    print(chunk, end='')
```

### How to use the Recommender

Initialize the engine and pass user preference dictionaries.

```python
from RecommendationSystem.model_inference import load_model_and_artifacts, RecommendationEngine, get_landmark_data

model, encoders, categories, config = load_model_and_artifacts()
unique_landmarks = get_landmark_data() # Loads from CSV
engine = RecommendationEngine(model, categories, unique_landmarks)

user_input = {
    'user_budget': 'medium',
    'user_preferences': ['Museum', 'Pharaonic Site'],
    'user_travel_type': 'solo',
    'user_age': 25,
    'user_gender': 'Male'
}
recommendations = engine.get_recommendations(user_input, top_n=10)
```

---

## 4. Key Features Added (v2.0)

- **Strict Multilingualism**: The chatbot automatically detects 16 languages and validates its own output to prevent "language mixing."
- **Adaptive Blending**: The recommender uses an 80/20 popularity-to-preference blend for new users, solving the "cold-start" problem.
- **Feedback Loop**: User "likes" or "dislikes" can be logged via `engine.log_interaction()` to improve future results.

---
**Ready for Deployment.**
*Prepared by AI Assistant for graduation project handoff.*


****NOTE‼️****

**System Requirements**

- pandas
- numpy
- requests
- faiss-cpu
- tensorflow
- scikit-learn
- ollama

-------------------------------------